# LLM assistant benchmark: how it is built, and how we know it works

Notebook 12 builds a research assistant: a large language model (DeepSeek) that answers plain-English questions about the CB2 dataset. This notebook is the evidence behind it. It explains **how the assistant is designed**, **how it was tested**, and **what the tests found**, including the question that started this work:

> Does DeepSeek need a chemistry-specialist model alongside it?

Everything shown here is read from saved results, so the notebook runs in seconds, needs no API key and costs nothing. The scripts that produced the results are named in each section so any number can be regenerated.

**Contents**

1. The design in one picture: the model reasons, code computes
2. Pre-fetch: the numbers arrive with the question
3. How the tests work: answers no language model wrote
4. Two modes, and a decision rule written before the results
5. Scoring that cannot be flattered
6. Time limits and forced answers
7. Results: each refinement, measured
8. Verdict, limitations, and how to rerun

In [1]:
from pathlib import Path
import sys
# Find the project folder whether the notebook runs from llm_assistant/ or the project root.
project_root = Path.cwd().resolve()
if project_root.name == "llm_assistant":
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "scripts"))
import json

import pandas as pd

import research_assistant as assistant
import chemistry_evaluation as chemistry

# Loading the dataset, fingerprints and the tuned SVR takes a few seconds. No language model is
# contacted anywhere in this notebook: every LLM answer below was saved by an earlier run.
data = assistant.ResearchData(project_root)
results = project_root / "results"
print(f"Loaded {len(data.compounds):,} curated molecules and {len(assistant.TOOL_SCHEMAS)} tools.")

Loaded 3,576 curated molecules and 21 tools.


## 1. The design in one picture: the model reasons, code computes

A language model is good at reading a question, planning, combining facts and explaining. It is bad at exact work: counting atoms in a SMILES string, reading the right row of a table, doing statistics. It does that work slowly when it does it at all, and it states wrong answers as confidently as right ones.

So the assistant splits the work. **DeepSeek never computes a number about a molecule or the dataset.** It decides which number it needs and asks for it; ordinary Python (pandas, RDKit, the saved SVR) computes it exactly and sends it back.

```
                  ┌──────────────── code, no language model ─────────────────┐
question ──▶ PRE-FETCH: find molecules named in the question, look them up ──┐
                                                                             ▼
                                         DeepSeek reads question + results, plans
                                                    │            ▲
                                  "call describe_molecule(...)"  │ exact result as JSON
                                                    ▼            │
                                    TOOLS: pandas · RDKit · tuned SVR · project files
                                                                             │
answer ◀── DeepSeek writes the answer, labelling each claim measured / predicted / general knowledge
```

| Job | Done by | Why |
| --- | --- | --- |
| Understand the question, choose what to look up | DeepSeek | Language understanding and planning |
| Formula, rings, stereocentres, R/S labels, properties | RDKit | Exact; DeepSeek takes minutes and still slips (section 7) |
| Counts, filters, group statistics, intervals | pandas / NumPy | Exact, with confidence intervals |
| pKi predictions and their uncertainty | The tuned SVR from notebook 07 | The model we validated; refuses reserved test molecules |
| Pharmacology, mechanism, explanation | DeepSeek | General knowledge; labelled as such in the answer |

A second language model to run the tools was considered and rejected: it would add a full model call to every question, and notebook 12 measured that smaller models are the weak link at choosing tools (two local models scored 9/18 and 6/18 where DeepSeek scored 18/18).

## 2. Pre-fetch: the numbers arrive with the question

Every tool request is a round trip to DeepSeek of several seconds. Many questions name their molecule outright, so asking DeepSeek to request the obvious look-up wastes a round. **Pre-fetch** finds SMILES strings, ChEMBL IDs and compound names with plain code (regular expressions and RDKit), runs the same tools DeepSeek would call, and attaches the results to the question. No language model is involved, and the results are exactly what the tool would have returned.

The cell below shows what pre-fetch does with a few questions. Notice what it ignores: ordinary words that happen to be valid SMILES ("CO2", "CCO") and dataset-wide questions with no molecule in them.

In [2]:
examples = [
    "What is the curated pKi of nabilone?",
    "What is the molecular formula of CCCCCCC(C)(C)c1cc(O)c2c(c1)OC(C)(C)C1CCC(=O)CC21?",
    "Compare methanandamide with CHEMBL600647.",
    "Why was cannabidiol removed during curation?",
    "How many molecules have a pKi of 9 or higher? Is CO2 or CCO relevant?",
]
# prefetch_requests is pure code: it returns the tool calls it would run, without running them.
print(f"{'Question':<62} | Look-ups attached before DeepSeek sees it")
print("-" * 130)
for question in examples:
    requests = assistant.prefetch_requests(data, question)
    shown = ", ".join(f"{name}({next(iter(arguments.values()))[:22]})" for name, arguments in requests) or "(none)"
    print(f"{question[:62]:<62} | {shown}")

Question                                                       | Look-ups attached before DeepSeek sees it
----------------------------------------------------------------------------------------------------------------------------------
What is the curated pKi of nabilone?                           | lookup_compound(CHEMBL2218896)
What is the molecular formula of CCCCCCC(C)(C)c1cc(O)c2c(c1)OC | describe_molecule(CCCCCCC(C)(C)c1cc(O)c2)
Compare methanandamide with CHEMBL600647.                      | lookup_compound(CHEMBL600647), lookup_compound(CHEMBL120526)
Why was cannabidiol removed during curation?                   | lookup_compound(CHEMBL190461)
How many molecules have a pKi of 9 or higher? Is CO2 or CCO re | (none)


Names are matched as whole words, longest first, and looked up **by ChEMBL ID** rather than by name. Building this exposed a real bug: the existing name lookup matched by "contains", so asking for *Ibipinabant* returned *Rac-Ibipinabant*, a different record. Exact names now win, and a test pins both cases.

Pre-fetched look-ups appear in the app's tool panel marked "(pre-fetched)", so a user can still see every number's source.

## 3. How the tests work: answers no language model wrote

A chat that sounds convincing is not evidence that it is correct. Every test question therefore has an answer computed **independently** of the assistant: by RDKit or pandas directly, taken from ChEMBL's records, or cited to a primary paper. The assistant answers in a fresh conversation and must end with `FINAL ANSWER: ...`, which is compared with the known answer.

Two question sets cover two different risks:

| Set | Script | Questions | What it checks |
| --- | --- | --- | --- |
| Dataset | `scripts/assistant_evaluation.py` | 18 | Reports the *data* correctly: counts, look-ups, statistics, provenance, the test-set refusal, saying "not in data" |
| Chemistry | `scripts/chemistry_evaluation.py` | 55 | The *chemistry* DeepSeek brings: what a specialist model would replace |

The chemistry set spans seven categories. Where each answer comes from:

In [3]:
# Build the chemistry questions exactly as the evaluation does: answers come from RDKit, ChEMBL
# records and cited papers, never from a language model.
questions = chemistry.build_questions(data)
frame = pd.DataFrame(questions)

print(f"{'Category':<18} | {'Questions':>9} | {'Where the known answer comes from':<52} | Could a tool answer it?")
print("-" * 140)
for category, group in frame.groupby("category", sort=False):
    sources = sorted(set(group["source"]))
    source = sources[0] if len(sources) == 1 else f"{len(sources)} sources, e.g. {sources[0]}"
    print(f"{category:<18} | {len(group):>9} | {source[:52]:<52} | {chemistry.TOOL_COULD_ANSWER[category]}")
print(f"\n{len(frame)} questions in total.")

Category           | Questions | Where the known answer comes from                    | Could a tool answer it?
--------------------------------------------------------------------------------------------------------------------------------------------
structure_reading  |         9 | 6 sources, e.g. RDKit CalcMolFormula                 | yes: describe_molecule computes it
cip_label          |         5 | RDKit new CIP labeller                               | yes: describe_molecule computes it (added after the first runs)
name_to_structure  |         8 | 8 sources, e.g. ChEMBL structure of CHEMBL122972     | partly: lookup_compound knows the 30 named dataset molecules only
structure_to_name  |         8 | 8 sources, e.g. ChEMBL preferred name of CHEMBL15848 | partly: lookup_compound knows the 30 named dataset molecules only
calculation        |         7 | 7 sources, e.g. Cheng & Prusoff 1973: Ki = IC50 / (1 | yes: arithmetic a tool could do
pharmacology       |        12 | 12 sources,

In [4]:
# One example per category, to show what the questions look like.
print(f"{'Category':<18} | {'Known answer':<16} | Question")
print("-" * 140)
for category, group in frame.groupby("category", sort=False):
    row = group.iloc[0]
    expected = "|".join(row["expected"]) if isinstance(row["expected"], tuple) else row["expected"]
    expected = f"{expected:.2f}" if isinstance(expected, float) else str(expected)
    print(f"{category:<18} | {expected[:16]:<16} | {row['question'][:100]}")

Category           | Known answer     | Question
--------------------------------------------------------------------------------------------------------------------------------------------
structure_reading  | C24H36O3         | What is the molecular formula of CCCCCCC(C)(C)c1cc(O)c2c(c1)OC(C)(C)C1CCC(=O)CC21?
cip_label          | R                | What is the CIP configuration (R or S) of the stereocentre in CCCCC/C=C\C/C=C\C/C=C\C/C=C\CCCC(=O)N[
name_to_structure  | CCCCC/C=C\C/C=C\ | Give a SMILES for anandamide, with stereochemistry if it has any.
structure_to_name  | ANANDAMIDE|ARACH | Which named compound (drug, natural product or endocannabinoid) has the structure CCCCC/C=C\C/C=C\C/
calculation        | 10.47            | A compound has pKi 7.98. What is its Ki in nM?
pharmacology       | A                | Which G protein family does the CB2 receptor couple to primarily? A) Gi/o B) Gq/11 C) Gs D) G12/13
medchem_concepts   | A                | Which group is the classic carbox

Two details keep the question set honest:

- **Molecules come from this project's data**, chosen by ChEMBL ID, so the structures are ChEMBL's and every structural answer is recomputed by RDKit when the questions are built.
- **Multiple-choice options are shuffled with a fixed seed.** Written by hand, most correct answers had landed on "B", and a model that favours one letter would then have scored well for the wrong reason. A test now fails if any letter is right for more than 40% of the questions.

## 4. Two modes, and a decision rule written before the results

Each chemistry question can be asked two ways:

- **Bare**: DeepSeek alone, with no tools and a one-line expert brief. This is DeepSeek's own chemistry: exactly what a specialist model would compete with. It was run on the first 11 questions to understand DeepSeek's behaviour (section 7d), then stopped: it showed what it needed to.
- **Assistant**: the deployed assistant, with the full system prompt, all tools and pre-fetch. This is what a user actually gets.

A low score only means something if we decided in advance what it would mean. The rule, coded in `decide()` before any result existed:

In [5]:
import inspect

# Print the decision rule's own documentation, so what is shown here is what the code applies.
print(inspect.getdoc(chemistry.decide))
print()
for category, could in chemistry.TOOL_COULD_ANSWER.items():
    print(f"  {category:<18} tool could answer: {could}")

Apply the decision rule to the deployed assistant's results, one line per category.

- accuracy at or above the threshold: no gap.
- a gap a deterministic tool could close: add or fix the tool; a specialist model is not needed.
- a gap only knowledge could close: a candidate for a specialist; score one with --mode bare first.

  structure_reading  tool could answer: yes: describe_molecule computes it
  cip_label          tool could answer: yes: describe_molecule computes it (added after the first runs)
  calculation        tool could answer: yes: arithmetic a tool could do
  name_to_structure  tool could answer: partly: lookup_compound knows the 30 named dataset molecules only
  structure_to_name  tool could answer: partly: lookup_compound knows the 30 named dataset molecules only
  pharmacology       tool could answer: no: literature knowledge
  medchem_concepts   tool could answer: no: textbook knowledge


The rule matters because "DeepSeek got it wrong" has two very different remedies. If it miscounts the rings in a SMILES string, the fix is a tool (RDKit counts exactly, every time); a specialist model would only miscount less often. A specialist model is justified only where **knowledge** is missing, which no tool can supply.

## 5. Scoring that cannot be flattered

An evaluation is only as strict as its scorer. The rules, each pinned by a test in `tests/test_chemistry_evaluation.py`:

- **Structures are compared by InChIKey**, a canonical identifier, so any valid SMILES of the right molecule scores and a near-miss does not. Stereochemistry is recorded separately.
- **Names must match, and near-misses are rejected.** Honokiol and magnolol are isomers; Δ8- and Δ9-THC differ by one double bond's position.
- **"UNKNOWN" is its own outcome**, neither right nor wrong: a model that knows when it does not know is safer than one that guesses.
- Numbers need the right value within a stated tolerance; a missing answer line counts as wrong.

The cell runs the real scorer on a few answers:

In [6]:
def first(category, text):
    # The first question in a category whose text contains a phrase.
    return next(q for q in questions if q["category"] == category and text in q["question"])

checks = [
    ("Δ8-THC from SMILES, answered 'Δ8-THC'",          "Δ8-THC",                          first("structure_to_name", "CC=C(C)C[C@@H]21")),
    ("Δ8-THC from SMILES, answered 'Δ9-THC'",          "DELTA-9-TETRAHYDROCANNABINOL",    first("structure_to_name", "CC=C(C)C[C@@H]21")),
    ("Honokiol, answered its methyl ether",            "4-O-METHYLHONOKIOL",              first("structure_to_name", "c(-c2ccc(O)c(CC=C)c2)")),
    ("Δ8-THC SMILES without stereo",                   "CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CC=C(C)CC21", first("name_to_structure", "delta-8")),
    ("Δ8-THC requested, Δ9-THC SMILES given",          "CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)=CC21", first("name_to_structure", "delta-8")),
    ("Ki from pKi 7.98, answered 10.5",                "10.5",                            first("calculation", "pKi 7.98")),
    ("Formula with subscripts",                        "C₂₄H₃₆O₃",                        first("structure_reading", "molecular formula")),
    ("Multiple choice, model says it cannot tell",     "UNKNOWN",                         first("pharmacology", "G protein")),
]
print(f"{'Case':<46} | {'Answer given':<42} | Scored")
print("-" * 105)
for label, answer, question in checks:
    print(f"{label:<46} | {answer[:42]:<42} | {chemistry.score(answer, question)}")

Case                                           | Answer given                               | Scored
---------------------------------------------------------------------------------------------------------
Δ8-THC from SMILES, answered 'Δ8-THC'          | Δ8-THC                                     | correct
Δ8-THC from SMILES, answered 'Δ9-THC'          | DELTA-9-TETRAHYDROCANNABINOL               | wrong
Honokiol, answered its methyl ether            | 4-O-METHYLHONOKIOL                         | wrong
Δ8-THC SMILES without stereo                   | CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CC=C(C)CC21   | correct
Δ8-THC requested, Δ9-THC SMILES given          | CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)=CC21   | wrong
Ki from pKi 7.98, answered 10.5                | 10.5                                       | correct
Formula with subscripts                        | C₂₄H₃₆O₃                                   | correct
Multiple choice, model says it cannot tell     | UNKNOWN                             

## 6. Time limits and forced answers

A language model has no clock: it does not know how long it has been thinking. During the first bare-mode run, one question kept DeepSeek reasoning for over four minutes with no end in sight. So the evaluation enforces a limit from outside, **120 seconds per question**, and when it runs out it does not throw the work away:

- **Bare mode** streams DeepSeek's private reasoning as it is written. At the limit the stream is closed, and DeepSeek is sent its own partial reasoning with: *"Time is up. Give your best answer now from that reasoning, without starting over. If it is not enough to decide, answer UNKNOWN."* Thinking is switched off for that reply, so it comes within seconds.
- **Assistant mode** stops starting new tool rounds once time is up and asks for an answer from the tool results gathered so far.

```python
for chunk in stream:                        # reasoning arrives a few words at a time
    reasoning.append(chunk.reasoning_content)
    if time.time() > deadline:
        stream.close()                      # stop thinking...
        break
# ...and answer from what was worked out:
forced = messages + [{"role": "user", "content": "Time is up. This is your reasoning so far: ..."}]
```

Every answer records `forced_by_time_limit`, so a slow correct answer can be told apart from a forced guess.

**A known limitation.** In assistant mode the limit is checked between tool rounds, so a request already running can overrun it: the worst case seen was 233 seconds. It was left as is deliberately. The limit exists for benchmarking; the app has none, so a genuinely hard question can take the time it needs.

## 7. Results: each refinement, measured

The assistant was refined in stages, and each stage was kept only if it made answers faster or more correct without breaking anything. To compare stages fairly they were **re-run side by side at the same time** (`scripts/design_stages.py`), each recreated without changing the app's code. That matters because DeepSeek's speed varies with its load: the same questions ran several times slower in the afternoon than in the morning, so stages measured hours apart would partly be measuring the hour.

| Stage | What changed | Why |
| --- | --- | --- |
| Earlier, notebook 12 | Schema in the prompt, smaller tool results, `find_linkers` | The first evaluation showed rounds wasted looking up column names |
| 1. Original | The design at the start of this work | Baseline |
| 2. + pre-fetch | Molecules named in a question are looked up before DeepSeek sees it | Saves a round trip when the look-up is the answer |
| 3. + R/S tool | `describe_molecule` reports R/S and E/Z labels from RDKit | DeepSeek took 1-3 minutes per R/S question by reasoning |
| 4. + "unspecified" note (current) | One sentence explaining unspecified stereocentres | Stage 3 was seen detouring, and erring, on them (section 7b) |

The chemistry stages use a fixed 20-question subset that covers every category; the dataset stages use all 18 dataset questions (which never involve R/S, so stages 2 and 3 are the same code there).

### 7a. The progression

In [7]:
stages = pd.read_csv(results / "assistant_design_stages.csv")
labels = {"1-original": "1. Original", "2-prefetch": "2. + pre-fetch", "3-final": "3. + R/S tool"}

for question_set in ["chemistry", "dataset"]:
    rows = stages[stages["question_set"] == question_set]
    print(f"{question_set.capitalize()} questions ({rows['number'].nunique()} questions, every stage run at the same time)")
    print(f"{'Stage':<24} | {'Correct':>7} | {'Unknown':>7} | {'Mean s':>6} | {'Median s':>8} | {'Slowest s':>9} | {'Total s':>7} | {'Cost':>6}")
    print("-" * 95)
    for stage, group in rows.groupby("stage"):
        unknown = (group["outcome"] == "unknown").sum() if "outcome" in group else 0
        print(f"{labels[stage]:<24} | {f'{group.correct.sum()}/{len(group)}':>7} | {unknown:>7} | {group.seconds.mean():>6.1f} | "
              f"{group.seconds.median():>8.1f} | {group.seconds.max():>9.0f} | {group.seconds.sum():>7.0f} | {'$' + format(group.cost_usd.sum(), '.2f'):>6}")
    print()

# Where the dataset questions started: notebook 12's own record of its first runs (measured on an earlier day).
history = pd.read_csv(results / "assistant_model_trials.csv")
print("Earlier history of the dataset questions (notebook 12, measured on an earlier day, so not directly comparable):")
for run, group in history[history["runs_on"] == "hosted"].groupby("run", sort=False):
    if "compact" in run:
        continue   # The compact-profile control is a different experiment, not a design stage.
    print(f"  {run:<50} {group.correct.sum()}/{len(group)} correct, {group.seconds.mean():.1f} s per question, "
          f"slowest {group.seconds.max():.0f} s")

Chemistry questions (20 questions, every stage run at the same time)
Stage                    | Correct | Unknown | Mean s | Median s | Slowest s | Total s |   Cost
-----------------------------------------------------------------------------------------------
1. Original              |   18/20 |       2 |   49.2 |     19.5 |       200 |     985 |  $0.41
2. + pre-fetch           |   20/20 |       0 |   35.0 |      9.1 |       155 |     699 |  $0.28
3. + R/S tool            |   20/20 |       0 |   12.4 |      8.9 |        66 |     247 |  $0.05

Dataset questions (18 questions, every stage run at the same time)
Stage                    | Correct | Unknown | Mean s | Median s | Slowest s | Total s |   Cost
-----------------------------------------------------------------------------------------------
1. Original              |   18/18 |       0 |   17.6 |     17.1 |        28 |     316 |  $0.07
2. + pre-fetch           |   18/18 |       0 |   17.5 |     15.8 |        34 |     314 |  $0.07

In [8]:
# Where the time went: chemistry mean seconds per category at each stage.
chemistry_stages = stages[stages["question_set"] == "chemistry"]
by_category = chemistry_stages.pivot_table(index="category", columns="stage", values="seconds", aggfunc="mean")
correct = chemistry_stages.pivot_table(index="category", columns="stage", values="correct", aggfunc="sum")
counts = chemistry_stages[chemistry_stages["stage"] == "3-final"].groupby("category").size()
order = list(dict.fromkeys(chemistry_stages.sort_values("number")["category"]))

print(f"{'Category':<18} | {'n':>2} | " + " | ".join(f"{labels[s]:>22}" for s in by_category.columns))
print("-" * 100)
for category in order:
    cells = [f"{by_category.at[category, s]:>6.1f} s  ({int(correct.at[category, s])}/{counts[category]} right)" for s in by_category.columns]
    print(f"{category:<18} | {counts[category]:>2} | " + " | ".join(f"{c:>22}" for c in cells))

Category           |  n |            1. Original |         2. + pre-fetch |          3. + R/S tool
----------------------------------------------------------------------------------------------------
structure_reading  |  4 |    16.9 s  (4/4 right) |     8.2 s  (4/4 right) |     7.4 s  (4/4 right)
cip_label          |  5 |   130.1 s  (3/5 right) |   109.2 s  (5/5 right) |     9.4 s  (5/5 right)
name_to_structure  |  3 |    45.5 s  (3/3 right) |    18.8 s  (3/3 right) |    28.8 s  (3/3 right)
structure_to_name  |  3 |    27.0 s  (3/3 right) |     7.4 s  (3/3 right) |    11.7 s  (3/3 right)
calculation        |  2 |     9.8 s  (2/2 right) |     7.9 s  (2/2 right) |     9.2 s  (2/2 right)
pharmacology       |  2 |    11.1 s  (2/2 right) |     8.4 s  (2/2 right) |    11.2 s  (2/2 right)
medchem_concepts   |  1 |     7.9 s  (1/1 right) |     9.0 s  (1/1 right) |     8.2 s  (1/1 right)


In [9]:
# Every answer that was not correct at any stage: what each refinement fixed, and anything it broke.
misses = stages[~stages["correct"]]
print(f"{'Stage':<24} | {'Set':<9} | {'Category':<18} | {'Outcome':<7} | {'Expected':<14} | {'Answered':<14} | Question")
print("-" * 140)
for row in misses.sort_values(["question_set", "number", "stage"]).itertuples():
    outcome = getattr(row, "outcome", "wrong") if isinstance(getattr(row, "outcome", None), str) else "wrong"
    print(f"{labels[row.stage]:<24} | {row.question_set:<9} | {str(getattr(row, 'category', '') or '')[:18]:<18} | "
          f"{outcome:<7} | {str(row.expected)[:14]:<14} | {str(row.answer)[:14]:<14} | {row.question[:36]}")
if misses.empty:
    print("(none: every stage answered every question correctly)")

Stage                    | Set       | Category           | Outcome | Expected       | Answered       | Question
--------------------------------------------------------------------------------------------------------------------------------------------
1. Original              | chemistry | cip_label          | unknown | S              | UNKNOWN        | What is the CIP configuration (R or 
1. Original              | chemistry | cip_label          | unknown | S,S            | UNKNOWN        | C[C@H](NC(=O)C(C)(C)Oc1ccc(C(F)(F)F)


What each refinement bought, on the same 20 chemistry questions run at the same time:

- **Pre-fetch (stage 1 → 2)** made the assistant faster *and* more reliable on questions that name a molecule: structure reading roughly halved (about 17 s to 8 s) and the name/structure questions fell from tens of seconds to under 20. It changed nothing on the dataset questions (17.6 s against 17.5 s), as expected: only 3 of the 18 name a molecule. It did not help the R/S questions at all, because the information DeepSeek needed was not in what was fetched.
- **The R/S tool (stage 2 → 3)** is the largest single gain. Without it, DeepSeek assigned R/S by reasoning: from 40 seconds to over 3 minutes per question, and in the original design it gave up twice ("UNKNOWN" at the time limit). With it, all five took 6 to 12 seconds and were right. Over the whole subset: **18/20 correct at 49 s per question, to 20/20 at 12 s**.
- **One side effect, found by these runs.** Nabilone is recorded as a racemate, so its two stereocentres are reported as "unspecified". Asked for nabilone's SMILES "with stereochemistry if it has any", DeepSeek on stage 3 started testing candidate stereoisomers itself (66 s here, 201 s in the full run) where the earlier stages answered in 20 to 34 s. That observation led to stage 4, below.
- The calculation, pharmacology and concept questions use no tools, so their times barely move between stages: a useful control showing the stages ran under the same conditions.

### 7b. Stage 4: a fix found by watching stage 3

The R/S tool reports a stereocentre as "unspecified" when the record does not define it, which is true of 821 of the 3,576 curated molecules (23%), mostly racemates. DeepSeek read "unspecified" as a puzzle to solve and set about building stereoisomers one tool call at a time. The candidate fix was one sentence in the tool's output, shown only for such molecules: *unspecified means the record does not define it; report it as unspecified, do not try to assign R or S.*

It was tested before being adopted, with a targeted A/B test rather than re-running every stage: 10 questions about molecules with unspecified stereocentres, each asked twice with the note (B) and without it (A), side by side (`scripts/stereo_note_ab_test.py`). The bar, set in advance: at least 30% faster on average, and no fewer correct.

In [10]:
ab = pd.read_csv(results / "stereo_note_ab_test.csv")
print(f"{'Variant':<22} | {'Correct':>7} | {'Mean s':>6} | {'Median s':>8} | {'Slowest s':>9} | {'Forced':>6}")
print("-" * 75)
for variant, label in [("A", "A: without the note"), ("B", "B: with the note")]:
    rows = ab[ab["variant"] == variant]
    print(f"{label:<22} | {f'{rows.correct.sum()}/{len(rows)}':>7} | {rows.seconds.mean():>6.1f} | {rows.seconds.median():>8.1f} | "
          f"{rows.seconds.max():>9.0f} | {rows.forced_by_time_limit.sum():>6}")

# The question that prompted the test, both passes.
nabilone = ab[ab["question"].str.startswith("Give a SMILES for nabilone")]
print("\nNabilone's SMILES, the question that prompted the test:")
for row in nabilone.sort_values(["variant", "repeat"]).itertuples():
    print(f"  {row.variant} pass {row.repeat}: {row.outcome:<7} {row.seconds:>6.1f} s, {row.n_tool_calls} tool calls")

Variant                | Correct | Mean s | Median s | Slowest s | Forced
---------------------------------------------------------------------------
A: without the note    |   18/20 |   23.6 |      7.5 |       149 |      2
B: with the note       |   20/20 |   16.0 |     11.0 |        58 |      0

Nabilone's SMILES, the question that prompted the test:
  A pass 1: wrong    149.2 s, 7 tool calls
  A pass 2: wrong    132.1 s, 7 tool calls
  B pass 1: correct   27.4 s, 1 tool calls
  B pass 2: correct   35.8 s, 1 tool calls


B passed the bar, but the result matters more for correctness than for speed. Without the note, DeepSeek got nabilone **wrong in both passes**, after more than two minutes each: building stereoisomers, it once moved a hydroxyl group and once shortened the side chain. With the note it was right both times in about half a minute. On the other questions the two variants were within a few seconds of each other, inside run-to-run noise. The note only ever appears for molecules with an unspecified stereocentre, so every other question is untouched. It was adopted, and a test pins it.

The full run below was made before stage 4. Nabilone was its one slow question (201 s, still correct that time); the other 54 questions do not involve the note.

### 7c. The full run

With the design settled, the full question sets were run once on it: all 55 chemistry questions and all 18 dataset questions. This is the accepted result.

In [11]:
accepted = pd.read_csv(results / "chemistry_evaluation.csv")
accepted = accepted[accepted["mode"] == "assistant"]
table = (accepted.groupby("category", sort=False)
         .agg(asked=("correct", "size"), correct=("correct", "sum"), mean=("seconds", "mean"),
              median=("seconds", "median"), slowest=("seconds", "max"),
              unknown=("outcome", lambda s: (s == "unknown").sum())))
print(f"{'Category':<18} | {'Correct':>7} | {'Unknown':>7} | {'Mean s':>6} | {'Median s':>8} | {'Slowest s':>9} | Could a tool answer it?")
print("-" * 125)
for category, row in table.iterrows():
    print(f"{category:<18} | {f'{int(row.correct)}/{int(row.asked)}':>7} | {int(row.unknown):>7} | {row['mean']:>6.1f} | {row['median']:>8.1f} | "
          f"{row.slowest:>9.0f} | {chemistry.TOOL_COULD_ANSWER[category]}")
print(f"\nChemistry: {accepted.correct.sum()}/{len(accepted)} correct, {accepted.seconds.mean():.1f} s per question, "
      f"{accepted.forced_by_time_limit.sum()} forced by the time limit, about ${accepted.cost_usd.sum():.2f}")

import assistant_evaluation
dataset = pd.read_csv(results / "assistant_evaluation.csv")
print(assistant_evaluation.summarise(dataset, "Dataset"))

Category           | Correct | Unknown | Mean s | Median s | Slowest s | Could a tool answer it?
-----------------------------------------------------------------------------------------------------------------------------
structure_reading  |     9/9 |       0 |    9.5 |      8.6 |        17 | yes: describe_molecule computes it
cip_label          |     5/5 |       0 |    6.5 |      6.5 |         8 | yes: describe_molecule computes it (added after the first runs)
name_to_structure  |     8/8 |       0 |   38.9 |     14.6 |       201 | partly: lookup_compound knows the 30 named dataset molecules only
structure_to_name  |     8/8 |       0 |   10.7 |     10.4 |        14 | partly: lookup_compound knows the 30 named dataset molecules only
calculation        |     7/7 |       0 |    6.8 |      6.7 |         8 | yes: arithmetic a tool could do
pharmacology       |   12/12 |       0 |    9.8 |      8.6 |        20 | no: literature knowledge
medchem_concepts   |     6/6 |       0 |    7.9 |  

In [12]:
# The decision rule from section 4, applied to the accepted run.
print(f"{'Category':<18} | {'Accuracy':>8} | {'Wrong':>5} | Verdict")
print("-" * 70)
for category, accuracy, wrong, verdict in chemistry.decide(accepted):
    print(f"{category:<18} | {accuracy:>8.0%} | {wrong:>5} | {verdict}")

misses = pd.concat([accepted[~accepted["correct"]].assign(set="chemistry"), dataset[~dataset["correct"]].assign(set="dataset")])
print(f"\nAnswers not correct in the accepted runs: {len(misses)}")
for row in misses.itertuples():
    print(f"  {row.set}: expected {row.expected}, answered {row.answer!r}: {row.question[:80]}")

Category           | Accuracy | Wrong | Verdict
----------------------------------------------------------------------
calculation        |     100% |     0 | no gap
cip_label          |     100% |     0 | no gap
medchem_concepts   |     100% |     0 | no gap
name_to_structure  |     100% |     0 | no gap
pharmacology       |     100% |     0 | no gap
structure_reading  |     100% |     0 | no gap
structure_to_name  |     100% |     0 | no gap

Answers not correct in the accepted runs: 0


### 7d. Why DeepSeek alone is slow: it reasons at length

Before settling on tools for structure work, DeepSeek was asked chemistry questions **with no tools at all**, to see what it knows by itself. It was right on all 11 questions it reached, but each took about 1.5 to 2 minutes, and that run was stopped early because the point was made. One question, nabilone's molecular formula, was then asked several ways to find out where the time goes:

In [13]:
development = pd.read_csv(results / "assistant_development_runs.csv")
bare = development[development["run"].str.startswith("Bare: DeepSeek alone")]
print(f"DeepSeek alone, no tools: {bare.correct.sum()}/{len(bare)} correct, {bare.seconds.mean():.0f} s per question, "
      f"{bare.forced_by_time_limit.sum()} stopped by the 120 s limit and forced to answer from their reasoning so far.\n")

reasoning = pd.read_csv(results / "deepseek_reasoning_length.csv")
print(f"{'Comparison':>10} | {'Model':<16} | {'Thinking':<8} | {'Prompt':<32} | {'Seconds':>7} | {'Output tokens':>13} | Answer")
print("-" * 120)
for row in reasoning.itertuples():
    verdict = "not recorded" if pd.isna(row.correct) else f"{row.answer} ({'correct' if row.correct else 'wrong'})"
    print(f"{row.comparison:>10} | {row.model:<16} | {'on' if row.thinking else 'off':<8} | {row.prompt:<32} | "
          f"{row.seconds:>7.1f} | {row.output_tokens:>13,} | {verdict}")

DeepSeek alone, no tools: 11/11 correct, 104 s per question, 7 stopped by the 120 s limit and forced to answer from their reasoning so far.

Comparison | Model            | Thinking | Prompt                           | Seconds | Output tokens | Answer
------------------------------------------------------------------------------------------------------------------------
         1 | deepseek-v4-pro  | on       | bare-mode brief + answer format  |   265.0 |        33,232 | not recorded
         1 | deepseek-v4-pro  | on       | plain question                   |   201.9 |        24,832 | not recorded
         1 | deepseek-v4-pro  | off      | plain question                   |    10.2 |         1,011 | not recorded
         1 | deepseek-flash   | on       | plain question                   |    61.4 |        13,151 | not recorded
         2 | deepseek-v4-pro  | on       | plain question                   |    95.1 |        11,647 | C24H36O3 (correct)
         2 | deepseek-v4-pro  | off 

What this shows:

- **The time is reasoning, not a slow server.** DeepSeek generates roughly 125 tokens a second; a formula took it 10,000 to 33,000 tokens of private reasoning, about the length of a short book chapter.
- **The prompt contributes, but is not the cause.** The evaluation's brief ("accuracy matters more than speed") added about a third; with no instructions at all it still reasoned for over three minutes.
- **The reasoning is what makes it right.** With thinking off, deepseek-v4-pro answered in 10 seconds and was wrong (C25H40O3). For exact work the choice is slow-and-right or fast-and-wrong; RDKit is fast *and* right, which is the case for tools.
- **A forced answer can be confidently wrong.** In a 20-second smoke test, DeepSeek was cut off mid-reasoning on EGCG and answered C22H18O10, one oxygen short. Answering "UNKNOWN" when unsure is encouraged, not guaranteed.
- Times vary a lot between identical requests (202 s and 95 s for the same question), so single requests are anecdotes. Comparison 1's correctness was not scored reliably (the check missed formulas written with subscripts) and is left blank rather than guessed.

### 7e. Other experiments along the way

The development runs record everything else that was tried. Two results shaped the final design:

In [14]:
def summary_of(run):
    rows = development[development["run"] == run]
    return f"{rows.correct.sum()}/{len(rows)} correct, {rows.seconds.mean():.1f} s per question"

print("Thinking off (dataset questions, run side by side in the morning):")
for run in ["Dataset B: pre-fetch on", "Dataset E: pre-fetch on, thinking off"]:
    print(f"  {run:<44} {summary_of(run)}, {development[development['run'] == run].n_tool_calls.mean():.1f} tool calls")

print("\nDoes DeepSeek find the R/S labels without pre-fetch handing them over?")
for run in ["R/S questions: R/S tool, pre-fetch on", "R/S questions: R/S tool, pre-fetch off"]:
    rows = development[development["run"] == run]
    print(f"  {run:<44} {summary_of(run)}; tools used: {', '.join(sorted(set(rows.tools)))}")

Thinking off (dataset questions, run side by side in the morning):
  Dataset B: pre-fetch on                      18/18 correct, 9.3 s per question, 1.3 tool calls
  Dataset E: pre-fetch on, thinking off        18/18 correct, 8.8 s per question, 1.7 tool calls

Does DeepSeek find the R/S labels without pre-fetch handing them over?
  R/S questions: R/S tool, pre-fetch on        5/5 correct, 7.6 s per question; tools used: describe_molecule (pre-fetched)
  R/S questions: R/S tool, pre-fetch off       5/5 correct, 15.0 s per question; tools used: describe_molecule


- **Thinking off barely changed the speed** (8.8 s against 9.3 s per question, within run-to-run noise) and DeepSeek made more tool calls without its planning step, as notebook 12 also measured. The app keeps thinking as a switch, on by default.
- **The R/S tool works even without pre-fetch.** DeepSeek called `describe_molecule` itself and used the labels rather than reasoning them out, so the tool's description does its job.

## 8. Verdict, limitations, and how to rerun

**Does DeepSeek need a chemistry-specialist model alongside it? No.** On the final design every chemistry category clears the 90% threshold set before the runs, so the decision rule finds no gap for a specialist to fill. The categories that rely on DeepSeek's own knowledge (CB2 pharmacology, medicinal-chemistry concepts, calculations) were answered correctly without tools; those that need exact structure work were answered by tools, quickly and exactly. Where DeepSeek was slow or wrong, the remedy was a tool, not a different model: the R/S labels and the note on unspecified stereocentres are the examples.

**What the tests cannot show:**

- **One run per stage, two passes in the A/B test.** Language models vary between runs; a repeat could miss a question these runs got right. `--repeats` in the chemistry evaluation asks everything several times.
- **A limited question set.** 55 chemistry and 18 dataset questions cover the kinds of question this assistant is asked, not all of chemistry. The pharmacology and concept questions are multiple choice, which is easier than open questions.
- **Speed depends on the provider.** Compare runs made side by side, not across days.
- **Faithful reporting is measured, not guaranteed.** The tools give DeepSeek exact values; the evaluations measure how often it reports them faithfully.

**To rerun** (each run calls DeepSeek and costs money; the scripts print the total):

```
python scripts/chemistry_evaluation.py --mode assistant                  # the full chemistry set on the final design
python scripts/assistant_evaluation.py --label "my change"               # the 18 dataset questions
python scripts/design_stages.py --stage 1-original --set chemistry --out stage1.csv   # one design stage
python scripts/chemistry_evaluation.py --mode bare                       # DeepSeek alone (slow, about an hour)
```

To test a candidate specialist model on exactly the same questions, point the chemistry evaluation at it:

```
python scripts/chemistry_evaluation.py --mode bare --base-url http://localhost:11434/v1 --model <specialist>
```